In [75]:
# Cargamos Pandas
import pandas as pd

In [76]:
# Se detecta que falta encabezados en clientes
clientes = pd.read_csv('clientes.csv')
clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 142 entries, 0 to 141
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     142 non-null    str  
 1   nombre         142 non-null    str  
 2   apellido       142 non-null    str  
 3   ciudad         142 non-null    str  
 4   codigo_postal  142 non-null    str  
 5   telefono       142 non-null    str  
 6   email          142 non-null    str  
dtypes: str(7)
memory usage: 7.9 KB


In [77]:
# Corregir encabezados de clientes
clientes = pd.read_csv('clientes.csv', header=None)

# Asignar encabezados correctos
clientes.columns = [
    'id_cliente',
    'nombre',
    'apellido',
    'ciudad',
    'codigo_postal',
    'telefono',
    'email'
]

In [78]:
# Se detecta que falta encabezados en ventas_accesorios
ventas = pd.read_csv('ventas_accesorios.csv', header=None)

# Asignar encabezados correctos
ventas.columns = [
    'id_venta',
    'fecha',
    'id_cliente',
    'producto',
    'categoria',
    'precio',
    'canal'
]

# Guardar limpio
ventas.to_csv('ventas_accesorios.csv', index=False)

In [79]:
# Cargamos los csv
encuesta = pd.read_csv('encuesta_sorteo.csv')
repuestos = pd.read_csv('repuestos.csv')
tickets = pd.read_csv('tickets_reparacion.csv')

In [80]:
# Normalizar nombres de columnas
def normalizar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("á","a")
        .str.replace("é","e")
        .str.replace("í","i")
        .str.replace("ó","o")
        .str.replace("ú","u")
    )
    return df

clientes = normalizar_columnas(clientes)
tickets = normalizar_columnas(tickets)
encuesta = normalizar_columnas(encuesta)
repuestos = normalizar_columnas(repuestos)
ventas = normalizar_columnas(ventas)

In [81]:
# Quitar filas basura 
clientes = clientes[clientes['codigo_postal'] != 'codigo_postal']
ventas = ventas[ventas['fecha'] != 'fecha']

In [82]:
# Convertir fechas
tickets['fecha'] = pd.to_datetime(tickets['fecha'])
ventas['fecha'] = pd.to_datetime(ventas['fecha'], format='mixed', dayfirst=True)
encuesta['fecha'] = pd.to_datetime(encuesta['fecha'])

In [83]:
# Convertir tipos 
clientes['codigo_postal'] = clientes['codigo_postal'].astype(int)
clientes['telefono'] = clientes['telefono'].astype(str)

ventas['precio'] = ventas['precio'].astype(int)

encuesta['edad'] = encuesta['edad'].astype(int)
encuesta['tiempo_espera'] = encuesta['tiempo_espera'].astype(int)
encuesta['calificacion'] = encuesta['calificacion'].astype(int)

In [84]:
# Eliminar duplicados
clientes.drop_duplicates(inplace=True)
tickets.drop_duplicates(inplace=True)
ventas.drop_duplicates(inplace=True)
encuesta.drop_duplicates(inplace=True)
repuestos.drop_duplicates(inplace=True)

In [85]:
# Unificar tipos para merge
clientes['id_cliente'] = clientes['id_cliente'].astype(int)
tickets['id_cliente'] = tickets['id_cliente'].astype(int)

In [86]:
# Merge final 
tickets_clientes = tickets.merge(clientes, on='id_cliente', how='left')
tickets_clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_ticket                250 non-null    int64         
 1   fecha                    250 non-null    datetime64[us]
 2   hora                     250 non-null    str           
 3   id_cliente               250 non-null    int64         
 4   modelo                   250 non-null    str           
 5   falla                    250 non-null    str           
 6   estado                   250 non-null    str           
 7   tiempo_reparacion_horas  250 non-null    float64       
 8   costo_repuesto           250 non-null    int64         
 9   precio_final             250 non-null    int64         
 10  tecnico                  250 non-null    str           
 11  canal                    250 non-null    str           
 12  nombre                   250 non-null    str   

In [87]:
# Top 10 fallas más frecuentes
tickets_clientes['falla'].value_counts().head(10)

falla
Software             40
Pantalla rota        37
Batería              36
Conector de carga    34
Altavoz              33
Humedad              26
Cámara               23
Micrófono            17
Placa base            4
Name: count, dtype: int64

In [88]:
# Top 10 modelos más reparados
tickets_clientes['modelo'].value_counts().head(10)

modelo
Samsung S21             27
Samsung S22             27
iPad 8                  26
Xiaomi Redmi Note 10    23
iPhone 11               22
iPhone XR               22
iPhone 12               21
iPhone SE 2020          21
iPhone 13               21
Xiaomi Redmi Note 11    17
Name: count, dtype: int64

In [89]:
# Tiempo promedio de reparación
tickets_clientes['tiempo_reparacion_horas'].mean()

np.float64(2.476)

In [90]:
# Cálculo del margen por reparación
tickets_clientes['margen'] = tickets_clientes['precio_final'] - tickets_clientes['costo_repuesto']
tickets_clientes['margen'].describe().round(2)


count    250.00
mean      38.05
std       11.57
min       20.00
25%       30.00
50%       33.00
75%       47.00
max       70.00
Name: margen, dtype: float64

In [91]:
# Rendimiento por técnico
tickets_clientes.groupby('tecnico').agg({
    'id_ticket':'count',
    'tiempo_reparacion_horas':'mean',
    'precio_final':'sum'
})


,id_ticket,tiempo_reparacion_horas,precio_final
tecnico,,,
Juan,50,2.43,3185
Lucía,50,2.63,3340
Marcos,50,2.39,3120
Pedro,50,2.55,3295
Sofía,50,2.38,3125


In [92]:
# Distribución por canal de ingreso
tickets_clientes['canal'].value_counts()


canal
Instagram        55
Tienda física    54
Google           53
Web              52
WhatsApp         36
Name: count, dtype: int64

In [93]:
#  Distribución por ciudad
tickets_clientes['ciudad'].value_counts()


ciudad
Godoy Cruz    63
Guaymallén    55
Las Heras     54
Maipú         44
Mendoza       34
Name: count, dtype: int64

In [94]:
# Margen promedio por ciudad
tickets_clientes.groupby('ciudad')['margen'].mean().sort_values(ascending=False).round(2)

ciudad
Guaymallén    43.22
Las Heras     40.13
Godoy Cruz    38.92
Mendoza       32.29
Maipú         32.25
Name: margen, dtype: float64

In [95]:
# Fallas por ciudad
tickets_clientes.groupby(['ciudad', 'falla'])['id_ticket'].count()


ciudad      falla            
Godoy Cruz  Altavoz               6
            Batería              15
            Conector de carga     6
            Cámara               15
            Humedad               9
            Micrófono             1
            Pantalla rota         3
            Placa base            1
            Software              7
Guaymallén  Altavoz               8
            Batería               5
            Conector de carga     7
            Cámara                5
            Humedad              12
            Micrófono             7
            Pantalla rota         8
            Placa base            2
            Software              1
Las Heras   Altavoz               6
            Batería              10
            Conector de carga    12
            Cámara                1
            Humedad               4
            Micrófono             2
            Pantalla rota        15
            Placa base            1
            Software              

In [96]:
# Modelos por ciudad
tickets_clientes.groupby(['ciudad', 'modelo'])['id_ticket'].count()


ciudad      modelo              
Godoy Cruz  MacBook Air 2017         2
            Samsung S21             14
            Samsung S22             13
            Xiaomi Redmi Note 10     3
            Xiaomi Redmi Note 11     1
            iPad 8                  10
            iPhone 11                4
            iPhone 12                5
            iPhone 13                1
            iPhone SE 2020           5
            iPhone XR                5
Guaymallén  MacBook Air 2017         1
            MacBook Air 2019         2
            Samsung S21              4
            Samsung S22             12
            Xiaomi Redmi Note 10    10
            Xiaomi Redmi Note 11     3
            iPad 8                   5
            iPhone 11                6
            iPhone 12                1
            iPhone 13                5
            iPhone XR                6
Las Heras   MacBook Air 2017         8
            MacBook Air 2019         2
            Samsung S21        

In [97]:
# Tiempo promedio por ciudad
tickets_clientes.groupby('ciudad')['tiempo_reparacion_horas'].mean().round(2)


ciudad
Godoy Cruz    2.56
Guaymallén    3.05
Las Heras     2.75
Maipú         1.81
Mendoza       1.84
Name: tiempo_reparacion_horas, dtype: float64

In [98]:
# Ingresos totales por ciudad
tickets_clientes.groupby('ciudad')['precio_final'].sum().sort_values(ascending=False)


ciudad
Guaymallén    4270
Godoy Cruz    4180
Las Heras     3825
Maipú         2110
Mendoza       1680
Name: precio_final, dtype: int64

In [99]:
# Canales por ciudad 
tickets_clientes.groupby(['ciudad', 'canal'])['id_ticket'].count()

ciudad      canal        
Godoy Cruz  Google           10
            Instagram        21
            Tienda física    18
            Web               6
            WhatsApp          8
Guaymallén  Google            6
            Instagram        13
            Tienda física    15
            Web               8
            WhatsApp         13
Las Heras   Google           14
            Instagram         7
            Tienda física    13
            Web              17
            WhatsApp          3
Maipú       Google           15
            Instagram         7
            Tienda física     3
            Web              13
            WhatsApp          6
Mendoza     Google            8
            Instagram         7
            Tienda física     5
            Web               8
            WhatsApp          6
Name: id_ticket, dtype: int64

<h1 style="text-align:center; font-size:34px; margin-bottom:20px;">
    Conclusiones del Proyecto
</h1>

<p style="font-size:16px;">
El análisis integral de los datos de clientes, tickets de reparación, ventas y encuestas permitió identificar patrones clave sobre el funcionamiento del servicio técnico y el comportamiento comercial.
</p>

<h2 style="font-size:26px; margin-top:25px;">1. Volumen y rentabilidad por ciudad</h2>
<p style="font-size:15px;">
Godoy Cruz concentra el mayor volumen de reparaciones, siendo el principal centro operativo.<br>
Guaymallén, aunque con menor volumen, presenta el margen promedio más alto, lo que la convierte en la ciudad más rentable por ticket.<br>
Las diferencias entre ciudades sugieren oportunidades para redistribuir recursos o ajustar precios.
</p>

<h2 style="font-size:26px; margin-top:25px;">2. Fallas y modelos más frecuentes</h2>
<p style="font-size:15px;">
Las fallas más comunes se concentran en pocas categorías, mostrando patrones repetitivos en los dispositivos.<br>
Los modelos más reparados coinciden con los más vendidos, lo que confirma consistencia entre ventas y servicio técnico.
</p>

<h2 style="font-size:26px; margin-top:25px;">3. Desempeño de los técnicos</h2>
<p style="font-size:15px;">
Existen diferencias claras en cantidad de tickets atendidos, tiempo promedio de reparación e ingresos generados.<br>
Esto permite identificar técnicos de alto rendimiento y otros que podrían beneficiarse de capacitación o redistribución de tareas.
</p>

<h2 style="font-size:26px; margin-top:25px;">4. Canales de ingreso</h2>
<p style="font-size:15px;">
El canal presencial sigue siendo el más utilizado, pero los canales digitales muestran crecimiento.<br>
La distribución por canal varía según la ciudad, lo que abre oportunidades para campañas segmentadas.
</p>

<h2 style="font-size:26px; margin-top:25px;">5. Tiempos de reparación</h2>
<p style="font-size:15px;">
El tiempo promedio de reparación es estable, pero algunas ciudades presentan demoras superiores.<br>
Esto puede deberse a carga de trabajo, disponibilidad de repuestos o eficiencia operativa.
</p>

<h2 style="font-size:26px; margin-top:25px;">6. Margen y rentabilidad</h2>
<p style="font-size:15px;">
El margen por reparación varía según ciudad, técnico y tipo de falla.<br>
Las fallas de mayor complejidad generan mayor margen, pero también mayor tiempo de reparación.
</p>

<h2 style="font-size:28px; text-align:center; margin-top:40px;">
    Síntesis Final
</h2>

<p style="font-size:16px;">
El negocio presenta buen volumen, margen saludable y patrones claros que permiten tomar decisiones estratégicas.<br>
Las oportunidades principales están en optimizar tiempos de reparación, potenciar canales digitales, reforzar ciudades con mayor margen, capacitar técnicos con menor rendimiento y ajustar precios según tipo de falla y ciudad.
</p>
